In [ ]:
# Parameters
message_bits = "10"


In [ ]:
%matplotlib inline
import matplotlib
matplotlib.use('agg')
import warnings
warnings.filterwarnings('ignore')


In [ ]:

from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.visualization import circuit_drawer, plot_histogram, plot_bloch_multivector
from qiskit.quantum_info import Statevector, partial_trace
import matplotlib.pyplot as plt
import numpy as np

message = str(message_bits)
if message not in ["00", "01", "10", "11"]:
    message = "10"

print(f"Superdense Coding — Alice sending message: {message}")

qc = QuantumCircuit(2, 2)

# 1. Entanglement Distribution (Create Bell Pair)
qc.h(0)
qc.cx(0, 1)
qc.barrier()

# 2. Alice's Encoding (on Qubit 0)
if message == '01':
    qc.x(0)
elif message == '10':
    qc.z(0)
elif message == '11':
    qc.z(0)
    qc.x(0)
qc.barrier()

# 3. Transmission (Implicit)
# 4. Bob's Decoding
qc.cx(0, 1)
qc.h(0)
qc.barrier()

# 5. Measurement
qc.measure([0, 1], [0, 1])

fig = circuit_drawer(qc, output='mpl')
display(fig)
plt.close(fig)

simulator = Aer.get_backend('aer_simulator')
compiled = transpile(qc, simulator)
job = simulator.run(compiled, shots=1000)
result = job.result()
counts = result.get_counts()

print(f"Message received by Bob: {list(counts.keys())[0]}")
print(f"Results: {counts}")

fig2 = plot_histogram(counts)
display(fig2)
plt.close(fig2)

# Bloch sphere (pre-measurement)
qc_sv = qc.remove_final_measurements(inplace=False)
sv = Statevector.from_instruction(qc_sv)
rho = partial_trace(sv, [1])
a = np.sqrt(np.real(rho.data[0, 0]))
b_complex = rho.data[1, 0] / a if a > 1e-6 else 0
theta = 2 * np.arccos(np.clip(a, 0, 1))
phi = np.angle(b_complex) % (2 * np.pi)

try:
    fig3 = plot_bloch_multivector(sv)
    display(fig3)
    plt.close(fig3)
except Exception:
    pass
print(f"BLOCH_THETA={theta:.6f}")
print(f"BLOCH_PHI={phi:.6f}")
